In [1]:
import os
import json
import numpy as np
import requests
from dotenv import load_dotenv
from pprint import pprint
import glob
from pypdf import PdfReader

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

if not API_KEY:
    print("No API key found! Copy .env.example to .env and put your key in it.")
else:
    print("API key loaded successfully.")

API key loaded successfully.


In [2]:
URL = "https://openrouter.ai/api/v1"

EMBEDDINGS_MODEL = "perplexity/pplx-embed-v1-0.6b"
CHAT_MODEL  = "google/gemini-3.8-flash"

CHUNK_SIZE = 150
TOP_K      = 5
MAX_TOKENS = 1000

HEADERS = {
    "Authorization": "Bearer " + API_KEY,
    "Content-Type": "application/json",
}

print("Embedding model:", EMBEDDINGS_MODEL)
print("Chat model:     ", CHAT_MODEL)
print("Max tokens:     ", MAX_TOKENS)

Embedding model: perplexity/pplx-embed-v1-0.6b
Chat model:      google/gemini-3.8-flash
Max tokens:      1000


In [3]:
response = requests.post(
        URL + "/chat/completions",
        headers=HEADERS,
        json={
            "model": CHAT_MODEL,
            "messages": [{"role": "user", "content": "What is Kenya's official policy on Ebola?"}],
            "max_tokens": 20,
        },
    )

if response.status_code == 200:
    print(response.json())
else:
    print("Something went wrong:", response.status_code, response.text)

{'id': 'gen-1791880024-8lnd54m6SJIyUCVcXLHW', 'object': 'chat.completion', 'created': 1791880024, 'model': 'google/gemini-3.8-flash', 'provider': 'Google', 'system_fingerprint': None, 'service_tier': 'default', 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'length', 'native_finish_reason': 'MAX_TOKENS', 'message': {'role': 'assistant', 'content': None, 'refusal': None, 'reasoning': "**Exploring Kenya's Official**\n\nI'm currently delving into the specifics of Kenya's official designations, aiming to clarify the most pertinent ones for our ongoing analysis. The focus is on identifying and understanding the primary official classifications that shape the landscape we're examining.\n\n", 'reasoning_details': [{'type': 'reasoning.text', 'text': "**Exploring Kenya's Official**\n\nI'm currently delving into the specifics of Kenya's official designations, aiming to clarify the most pertinent ones for our ongoing analysis. The focus is on identifying and understanding the primary

In [4]:
pprint(response.json())

{'choices': [{'finish_reason': 'length',
              'index': 0,
              'logprobs': None,
              'message': {'content': None,
                          'reasoning': "**Exploring Kenya's Official**\n"
                                       '\n'
                                       "I'm currently delving into the "
                                       "specifics of Kenya's official "
                                       'designations, aiming to clarify the '
                                       'most pertinent ones for our ongoing '
                                       'analysis. The focus is on identifying '
                                       'and understanding the primary official '
                                       'classifications that shape the '
                                       "landscape we're examining.\n"
                                       '\n',
                          'reasoning_details': [{'format': 'google-gemini-v1',
           

In [5]:
pdf_files = glob.glob("corpus/*")

documents = []

for file_path in pdf_files:
    print(f"--- Opening: {file_path} ---")

    reader = PdfReader(file_path)

    text = []
    for page in reader.pages or []:
        text.append(page.extract_text())

    documents.append({"file_name": file_path, "text": "".join(text)})

--- Opening: corpus/Kenya_VMMC_Guidelines_2008.pdf ---
--- Opening: corpus/9789240008540-eng.pdf ---
--- Opening: corpus/09.04.2019%2520GP%2520Contracting%2520guidelines%2520v8_formatted.pdf ---
--- Opening: corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf ---
--- Opening: corpus/Making-voluntary-medical-male-circumcision-services-sustainable-Findings-from-Kenyas-pilot-models-baseline-and-year-1.pdf ---
--- Opening: corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf ---


In [6]:
[document["file_name"] for document in documents]

['corpus/Kenya_VMMC_Guidelines_2008.pdf',
 'corpus/9789240008540-eng.pdf',
 'corpus/09.04.2019%2520GP%2520Contracting%2520guidelines%2520v8_formatted.pdf',
 'corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf',
 'corpus/Making-voluntary-medical-male-circumcision-services-sustainable-Findings-from-Kenyas-pilot-models-baseline-and-year-1.pdf',
 'corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf']

In [7]:
documents[3]["text"][:3000]

'VOLUNTARY MEDICAL MALE CIRCUMCISION \nIN-SERVICE COMMUNICATION\nBEST PRACTICES GUIDE\n1\nVoluntary Medical Male Circumcision In-Service Communication: Best Practices Guide\nACKNOWLEDGEMENTS\nThis reference guide is the result of collaboration among the members of the Communication Sub Group \n(CSG) of the U.S. President’s Emergency Plan for AIDS Relief (PEPFAR) Voluntary Medical Male Circumcision \n(VMMC) Technical Working Group (TWG), which includes Kim Seifert-Ahanda, Jonathan Davitte, Elizabeth \nGold, Dan Rutz and Catey Laube, and the Health Communication Capacity Collaborative (HC3). \nSpecial thanks go to Karin Hatzold of PSI Zimbabwe, Jason Reed of Jhpiego/AIDSFree and John Byabagambi \nof URC/ASSIST for their contributions, as well as our colleagues from UNICEF, WHO, USAID, CDC, OGAC, and \nJhpiego/AIDSFree who reviewed the guide and provided valuable input. Special recognition goes to Katie \nKuehn and Anna Ellis of HC3 for their copyedit and design work.2\nFOREWORD\nThe U.S.

In [8]:
def make_chunks(text, size=CHUNK_SIZE):
    words = text.split()
    chunks = []
    for start in range(0, len(words), size):
        piece = words[start:start + size]
        chunks.append(" ".join(piece))
    return chunks

In [9]:
chunks = []

for doc in documents:
    for piece in make_chunks(doc["text"]):
        chunks.append({"source": doc["file_name"], "text": piece})

print("The", len(documents), "documents became", len(chunks), "chunks")
print("\nHere is chunk number 5:\n")
print("from:", chunks[5]["source"])
print("text:", chunks[5]["text"], "...")

The 6 documents became 783 chunks

Here is chunk number 5:

from: corpus/Kenya_VMMC_Guidelines_2008.pdf
text: on male circumcision and HIV prevention in March 2007. They stated that, "the efficacy of male circumcision in reducing female to male HIV transmission has now been proven beyond reasonable doubt," and that male circumcision should.be considered as part of a comprehensive HIV prevention package6. In recognition of the above recommendations the Ministry of Health has developed this guidance document for male circumcision in Ke- nya. The guidance document will provide a framework to ensure safe, accessible, sustainable male circumcision services through guidance to policy makers and implementers. National Guidance for Voluntary Male Circumcision in Kenya 1SITUATION ANALYSIS Male circumcision is practiced by many communities in Kenya. In ad- dition to religious reasons, circumcision often serves as a rite of pas- sage. Data from the 2003 Kenyan Demographic and Health Survey shows 

In [10]:
def get_embeddings(texts, embeddings_model=EMBEDDINGS_MODEL):
    response = requests.post(
                URL + "/embeddings",
                headers=HEADERS,
                json={"model": embeddings_model, "input": texts},
            )

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return None

    results = response.json()["data"]
    vectors = [item["embedding"] for item in results]
    return np.array(vectors)

In [11]:
def get_embeddings_in_batches(chunks, embeddings_model=EMBEDDINGS_MODEL):
    chunk_texts_first_batch = [c["text"] for c in chunks[:400]]
    chunk_texts_second_batch = [c["text"] for c in chunks[400:]]
    chunk_vectors_first_batch = get_embeddings(chunk_texts_first_batch, embeddings_model=embeddings_model)
    chunk_vectors_second_batch = get_embeddings(chunk_texts_second_batch, embeddings_model=embeddings_model)

    return np.concatenate([chunk_vectors_first_batch, chunk_vectors_second_batch], axis=0)

In [12]:
get_embeddings(chunks[5]["text"])

array([[-0.0546875, -0.0859375,  0.       , ...,  0.3984375,  0.171875 ,
         0.1953125]], shape=(1, 1024))

In [13]:
chunk_vectors = get_embeddings_in_batches(chunks)

In [14]:
len(chunks)

783

In [15]:
chunk_vectors

array([[-0.0078125, -0.140625 ,  0.       , ...,  0.2265625,  0.046875 ,
        -0.0078125],
       [-0.0078125, -0.109375 ,  0.       , ...,  0.2578125,  0.15625  ,
         0.140625 ],
       [ 0.03125  , -0.140625 ,  0.       , ...,  0.1796875,  0.125    ,
        -0.0234375],
       ...,
       [-0.0078125,  0.0078125,  0.       , ...,  0.078125 ,  0.0703125,
         0.140625 ],
       [-0.0234375, -0.0546875,  0.       , ...,  0.1484375,  0.0625   ,
         0.1796875],
       [-0.09375  , -0.1640625,  0.       , ...,  0.203125 ,  0.0625   ,
         0.234375 ]], shape=(783, 1024))

In [16]:
def normalise(vectors):
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / lengths

In [17]:
chunk_vectors_normalised = normalise(chunk_vectors)

In [18]:
def search(question, k=TOP_K, chunk_vectors_normalised=chunk_vectors_normalised):
    question_vector = normalise(get_embeddings([question]))[0]

    scores = chunk_vectors_normalised @ question_vector

    best = np.argsort(scores)[::-1][:k]

    results = []
    for i in best:
        results.append({
            "source": chunks[i]["source"],
            "text":   chunks[i]["text"],
            "score":  round(float(scores[i]), 3),
        })
    return results

In [19]:
question = "What is VMMC good?"

found   = search(question,k=5)

print("QUESTION:", question)
print()
for i, hit in enumerate(found, 1):
    print(i, "| score:", hit["score"], "| from:", hit["source"])
    print("   ", hit["text"][:250], "...")
    print()

QUESTION: What is VMMC good?

1 | score: 0.535 | from: corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf
    for International Development VCT Voluntary Counseling and Testing VMMC Voluntary Medical Male Circumcision WHO World Health Organization PEPFAR’s Best Practices for Voluntary Medical Male Circumcision Site Operations v MESSAGE FROM THE US GLOBAL AID ...

2 | score: 0.532 | from: corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf
    without fear of interruption. Confidentiality Assure client that all discussions in this session are confidential. Key Information on VMMC • VMMC is different from traditional male circumcision. (This may not be relevant in all settings and culture s ...

3 | score: 0.523 | from: corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf
    as well as partners, guardians, and family members about the facts of VMMC and combination HIV prevention Primary Audiences: Poten

In [20]:
RULES = """You are a clinical assistant for a VMMC medical drive.

Answer the question using ONLY the corpus provided.

- Do not use any outside knowledge.
- Say which file each fact came from, like [corpus/Kenya_VMMC_Guidelines_2008.pdf].
- If the corpus does not contain the answer, reply exactly:
  "This is new knowledge not contained in the corpus."
- Keep the answer short - 4 or 5 sentences.
"""

In [21]:
def ask_ai(question, input_content):
    
    message = "CORPUS:\n" + input_content + "\n\nQUESTION: " + question

    response = requests.post(
        URL + "/chat/completions",
        headers=HEADERS,
        json={
            "model": CHAT_MODEL,
            "messages": [
                {"role": "system", "content": RULES},
                {"role": "user",   "content": message},
            ],
            "temperature": 0,
            "max_tokens": MAX_TOKENS,
        },
    )

    if response.status_code != 200:
        return "Error: " + response.text

    return response.json()["choices"][0]["message"]["content"]

In [22]:
def rag(question, k=TOP_K, chunk_vectors_normalised=chunk_vectors_normalised):
    
    found = search(question, k, chunk_vectors_normalised)
    
    content = ""
    for hit in found:
        content += "[" + hit["source"] + "]\n" + hit["text"] + "\n\n"

    answer = ask_ai(question, content)

    print("QUESTION:", question)
    print()
    print("ANSWER:")
    print(answer)
    print()
    print("(based on:", ", ".join(h["source"] for h in found), ")")

    return answer

In [23]:
rag("What is VMMC?")

QUESTION: What is VMMC?

ANSWER:
VMMC stands for Voluntary Medical Male Circumcision, which is a proven biomedical intervention involving the removal of the foreskin [corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf], [corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf]. It differs from traditional male circumcision and can be performed by surgery or by device [corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf]. The procedure is done primarily to reduce a man's risk of acquiring HIV through heterosexual intercourse by approximately 60%, providing partial protection [corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf], [corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf]. Additionally, VMMC provides a reduction of ulcerative ST

(based on: corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf, corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf, corpus/PEPFA

In [24]:
rag("When was the UN created?")

QUESTION: When was the UN created?

ANSWER:
This is new knowledge not contained in the corpus.

(based on: corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf )


In [25]:
rag("When did man go to the moon?")

QUESTION: When did man go to the moon?

ANSWER:
This is new knowledge not contained in the corpus.

(based on: corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf, corpus/9789240008540-eng.pdf )


# Comparing embedding models

In [26]:
embedding_models = [
    "baai/bge-m3",
    "mistralai/mistral-embed-2312",
    "liquid/lfm-2.5-embedding-350m:free",
    "intfloat/e5-large-v2",
    "thenlper/gte-large"
]
questions = ["What is VMMC?", "When was the UN created?", "When did man go to the moon?"]

In [28]:
for embedding in embedding_models:
    print("-"*100)
    print(f"Considering {embedding}")
    print("-"*100)
    try:
        chunk_vectors = get_embeddings_in_batches(chunks, embedding)
        chunk_vectors_normalised = normalise(chunk_vectors)
        for question in questions:
            rag(question, chunk_vectors_normalised=chunk_vectors_normalised)
    except Exception as exc:
        print(exc)
    print("\n")
    print("-"*100)
    print("\n")

----------------------------------------------------------------------------------------------------

 Considering baai/bge-m3
QUESTION: What is VMMC?

ANSWER:
Based on the provided corpus, VMMC stands for Voluntary Medical Male Circumcision [corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1

(based on: corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf, corpus/VMMC-Counseling-Guide-FINAL-8-15-16-1-1.pdf, corpus/09.04.2019%2520GP%2520Contracting%2520guidelines%2520v8_formatted.pdf, corpus/09.04.2019%2520GP%2520Contracting%2520guidelines%2520v8_formatted.pdf, corpus/PEPFARs_Best_Practices_for_Voluntary_Medical_Male_Circumcision_Site_Operations-2013.pdf )
QUESTION: When was the UN created?

ANSWER:
This is new knowledge not contained in the corpus.

(based on: corpus/9789240008540-eng.pdf, corpus/Making-voluntary-medical-male-circumcision-services-sustainable-Findings-from-Kenyas-pilot-models-baseline-and-year-1.pdf, corpus/9789240008540-eng.pdf, 